# ESM-2 for Enzyme Function Prediction

**Fully self-contained** — upload this `.ipynb` to Google Colab, enable GPU (`Runtime → Change runtime type → T4 GPU`), and run all cells.

Pipeline:
1. Install dependencies
2. Download enzyme data from UniProt (with GO annotations)
3. Explore the dataset
4. Extract ESM-2 embeddings
5. Train classifiers: sklearn baselines + MLP + Residual MLP
6. Train Hierarchical EC Classifier (DEEPre-inspired, level-by-level)
7. Train Multi-task EC + GO model (auxiliary GO head)
8. Generate all figures

Total runtime on Colab T4 GPU: ~90-120 minutes.

In [ ]:
# CELL 1: Install dependencies + verify GPU
!pip install fair-esm -q
!pip install pandas numpy scikit-learn matplotlib seaborn requests tqdm -q

import os, sys, torch, json
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

EC_CLASSES = [f"EC{i}" for i in range(1, 7)]
EC_NAMES = {
    1: "Oxidoreductases", 2: "Transferases", 3: "Hydrolases",
    4: "Lyases", 5: "Isomerases", 6: "Ligases"
}
MAX_SEQ_LENGTH = 1022
os.makedirs("results/figures", exist_ok=True)
os.makedirs("models", exist_ok=True)

## Step 1: Download Data (with GO annotations)

In [ ]:
# CELL 2: Download enzyme sequences from UniProt SwissProt
#
# Query: reviewed entries with EC number, seq length <= 1022 (ESM-2 limit)
# Fetches: accession, sequence, EC, protein name, organism, length, GO annotations
# GO terms are used by the Multi-task EC+GO model as auxiliary labels.
import requests
import time

def extract_ec(entry):
    """Extract EC numbers from proteinDescription.recommendedName.ecNumbers."""
    pd_ = entry.get("proteinDescription", {})
    ecs = []
    for section in ("recommendedName", "alternativeNames"):
        data = pd_.get(section)
        items = data if isinstance(data, list) else ([data] if isinstance(data, dict) else [])
        for item in items:
            for ec in item.get("ecNumbers", []):
                if ec.get("value"): ecs.append(ec["value"])
    return ecs

def extract_go(entry):
    """Extract GO IDs from uniProtKBCrossReferences (database=GO)."""
    return sorted({
        x["id"] for x in entry.get("uniProtKBCrossReferences", [])
        if x.get("database") == "GO" and str(x.get("id", "")).startswith("GO:")
    })

def query_uniprot(ec_class, max_results=1500):
    base_url = "https://rest.uniprot.org/uniprotkb/search"
    query = f"(reviewed:true) AND (ec:{ec_class}.*.*) AND (length:[* TO {MAX_SEQ_LENGTH}])"
    params = {
        "query": query, "format": "json",
        "fields": "accession,sequence,ec,protein_name,organism_name,length,go",
        "size": min(max_results, 500),
    }
    proteins, next_link = [], None
    while True:
        url = next_link or base_url
        resp = requests.get(url, params=params if not next_link else None, timeout=30)
        if resp.status_code != 200:
            print(f"  HTTP {resp.status_code} for EC{ec_class}"); break
        for entry in resp.json().get("results", []):
            seq = entry.get("sequence", {}).get("value", "")
            if not (seq and 20 <= len(seq) <= MAX_SEQ_LENGTH): continue
            ec_list = extract_ec(entry)
            go_ids = extract_go(entry)
            proteins.append({
                "accession": entry.get("primaryAccession", ""),
                "sequence": seq,
                "ec_class": ec_class,
                "ec_number": ec_list[0] if ec_list else f"{ec_class}.-.-.-",
                "ec_level1_name": EC_NAMES[ec_class],
                "protein_name": entry.get("proteinDescription", {}).get("recommendedName", {}).get("fullName", {}).get("value", "Unknown") if entry.get("proteinDescription", {}).get("recommendedName") else "Unknown",
                "organism": entry.get("organism", {}).get("scientificName", "Unknown"),
                "length": len(seq),
                "go_terms": ";".join(go_ids[:20]),
            })
        link = resp.headers.get("Link", "")
        if 'rel="next"' in link:
            next_link = link.split(";")[0].strip("<>"); params = None; time.sleep(0.5)
        else: break
        if len(proteins) >= max_results: break
    return proteins

all_proteins = []
for ec_class in range(1, 7):
    print(f"Downloading EC{ec_class} ({EC_NAMES[ec_class]})...")
    proteins = query_uniprot(ec_class, max_results=1500)
    all_proteins.extend(proteins)
    go_count = sum(1 for p in proteins if p["go_terms"])
    print(f"  Got {len(proteins)} sequences ({go_count} with GO annotations)")
    time.sleep(1)

df = pd.DataFrame(all_proteins)
df.to_csv("enzymes_swissprot.csv", index=False)
print(f"\nTotal: {len(df)} sequences saved to enzymes_swissprot.csv")
print(f"\nClass distribution:\n{df['ec_class'].value_counts().sort_index()}")
df.head()

## Step 2: Explore Dataset

In [ ]:
# CELL 3: Explore dataset
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

counts = df["ec_class"].value_counts().sort_index()
axes[0].bar(EC_CLASSES, counts.values, color=plt.cm.Set2(np.linspace(0, 1, 6)))
axes[0].set_title("EC Class Distribution")
axes[0].set_ylabel("Count")
for i, v in enumerate(counts.values): axes[0].text(i, v+10, str(v), ha="center", fontweight="bold")

axes[1].hist(df["length"], bins=50, color="#2196F3", edgecolor="white")
axes[1].axvline(df["length"].median(), color="red", linestyle="--", label=f'Median: {df["length"].median():.0f}')
axes[1].set_title("Sequence Length Distribution")
axes[1].set_xlabel("Length (aa)"); axes[1].legend()

go_counts = df["go_terms"].apply(lambda x: len(str(x).split(";")) if x else 0)
axes[2].hist(go_counts[go_counts > 0], bins=30, color="#4CAF50", edgecolor="white")
axes[2].set_title("GO Terms per Enzyme (where available)")
axes[2].set_xlabel("Number of GO terms")

plt.tight_layout()
plt.savefig("results/figures/dataset_overview.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 3: Extract ESM-2 Embeddings

In [ ]:
# CELL 4: Extract mean-pooled ESM-2 embeddings
#
# ESM-2 (35M params, 12 transformer layers, 480-dim) trained on 250M+ sequences.
# Per-residue embeddings are mean-pooled to get one 480-dim vector per protein.
# N_SAMPLES: set to a small number (e.g. 500) first to test speed, then set to len(df).
import esm

MODEL_NAME = "esm2_t12_35M_UR50D"
print("Loading ESM-2...")
esm_model, alphabet = esm.pretrained.load_model_and_alphabet(MODEL_NAME)
esm_model = esm_model.to(device).eval()
batch_converter = alphabet.get_batch_converter()
num_layers = getattr(esm_model, "num_layers", None) or getattr(getattr(esm_model, "args", None), "num_layers", 12)
embed_dim = getattr(esm_model, "embed_dim", None) or getattr(getattr(esm_model, "args", None), "embed_dim", 480)
print(f"  {embed_dim}-dim embeddings, {num_layers} layers | Device: {device}")

N_SAMPLES = len(df)  # set small to test, then full
seqs = df["sequence"].tolist()[:N_SAMPLES]
labels = (df["ec_class"].values - 1).astype(np.int64)[:N_SAMPLES]

def extract_embeddings(sequences, batch_size=32):
    all_embeddings = []
    for i in tqdm(range(0, len(sequences), batch_size), desc="Embeddings"):
        batch_seqs = sequences[i:i + batch_size]
        data = [(f"seq_{j}", s) for j, s in enumerate(batch_seqs)]
        _, _, tokens = batch_converter(data)
        tokens = tokens.to(device)
        with torch.no_grad():
            res = esm_model(tokens, repr_layers=[num_layers], return_contacts=False)
        reps = res["representations"][num_layers]
        for j in range(len(batch_seqs)):
            seq_len = len(batch_seqs[j])
            emb = reps[j, 1:seq_len + 1].mean(dim=0)
            all_embeddings.append(emb.cpu().numpy())
    return np.array(all_embeddings, dtype=np.float32)

print(f"Extracting embeddings for {len(seqs)} sequences...")
embeddings = extract_embeddings(seqs)
np.save("esm2_embeddings.npy", embeddings)
np.save("labels.npy", labels)
print(f"Embeddings: {embeddings.shape}  Labels: {labels.shape}")

# Free GPU memory
del esm_model; torch.cuda.empty_cache()

## Step 4: Split Dataset

In [ ]:
# CELL 5: Stratified train/val/test split (70/15/15)
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score

embeddings = np.load("esm2_embeddings.npy")
labels = np.load("labels.npy")

# Hierarchical labels: Level 1 = main class (0-5), Level 2 = subclass from ec_number
y1_all = (df["ec_class"].values - 1).astype(np.int64)[:N_SAMPLES]
def parse_subclass(ec):
    try:
        parts = str(ec).split(".")
        if len(parts) > 1 and parts[1].isdigit():
            return max(0, int(parts[1]) - 1)
    except (IndexError, ValueError):
        pass
    return 0
y2_all = np.array([parse_subclass(ec) for ec in df["ec_number"].values[:N_SAMPLES]], dtype=np.int64)
EC_SUB = {1: 28, 2: 26, 3: 13, 4: 8, 5: 6, 6: 7}  # subclass head widths per class
for cls, cnt in EC_SUB.items():
    m = (y1_all == cls - 1)
    y2_all[m] = np.clip(y2_all[m], 0, cnt - 1)  # keep in-range for CrossEntropyLoss

# One split, referenced everywhere — guarantees train/val/test alignment
all_idx = np.arange(len(labels))
trainval_idx, test_idx = train_test_split(all_idx, test_size=0.15, stratify=labels, random_state=42)
train_idx, val_idx = train_test_split(trainval_idx, test_size=0.176, stratify=labels[trainval_idx], random_state=42)

X_train, y_train = embeddings[train_idx], labels[train_idx]
X_val, y_val = embeddings[val_idx], labels[val_idx]
X_test, y_test = embeddings[test_idx], labels[test_idx]
y1_train, y2_train = y1_all[train_idx], y2_all[train_idx]
y1_val, y2_val = y1_all[val_idx], y2_all[val_idx]
y1_test, y2_test = y1_all[test_idx], y2_all[test_idx]

# GO labels: binary matrix over top 20 GO terms
TOP_GO = [
    "GO:0005524", "GO:0003674", "GO:0005488", "GO:0003824", "GO:0006468",
    "GO:0005737", "GO:0016020", "GO:0005886", "GO:0005515", "GO:0008270",
    "GO:0046872", "GO:0005634", "GO:0005829", "GO:0016021", "GO:0005783",
    "GO:0009986", "GO:0045087", "GO:0006915", "GO:0042981", "GO:0007165",
]
go_matrix = np.zeros((N_SAMPLES, len(TOP_GO)), dtype=np.float32)
term_idx = {t: i for i, t in enumerate(TOP_GO)}
for i, terms in enumerate(df["go_terms"].values[:N_SAMPLES]):
    if pd.isna(terms) or not terms: continue
    for t in str(terms).split(";"):
        t = t.strip()
        if t in term_idx: go_matrix[i, term_idx[t]] = 1.0
print(f"GO labels: {go_matrix.shape}, nonzero samples: {(go_matrix.sum(axis=1) > 0).sum()}")

y_go_train, y_go_val, y_go_test = go_matrix[train_idx], go_matrix[val_idx], go_matrix[test_idx]

print(f"Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}")

## Step 5: Sklearn Baselines

In [ ]:
# CELL 6: Logistic Regression, Random Forest, Gradient Boosting
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

baselines = {
    "Logistic Regression": LogisticRegression(max_iter=1000, C=1.0, solver="lbfgs", multi_class="multinomial"),
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=20, min_samples_leaf=5,
                                            class_weight="balanced", random_state=42, n_jobs=-1),
    "Gradient Boosting": HistGradientBoostingClassifier(max_iter=300, max_depth=5,
                                                        learning_rate=0.1, random_state=42),
}
sklearn_results = {}
for name, model in baselines.items():
    print(f"\nTraining {name}...")
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    sklearn_results[name] = {
        "accuracy": accuracy_score(y_test, preds),
        "f1_macro": f1_score(y_test, preds, average="macro"),
        "report": classification_report(y_test, preds, output_dict=True),
    }
    print(f"  Accuracy: {sklearn_results[name]['accuracy']:.4f}  F1: {sklearn_results[name]['f1_macro']:.4f}")

## Step 6: MLP Classifier

In [ ]:
# CELL 7: MLP: 480 → Linear(256) → BN → ReLU → Dropout → Linear(128) → BN → ReLU → Dropout → 6
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

class MLPClassifier(nn.Module):
    def __init__(self, input_dim=480, num_classes=6, dropout=0.3):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )
    def forward(self, x): return self.network(x)

def train_model(model, X_tr, y_tr, X_v, y_v, name="model", epochs=50, bs=64, lr=1e-3):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit = nn.CrossEntropyLoss()
    tr_loader = DataLoader(TensorDataset(torch.FloatTensor(X_tr), torch.LongTensor(y_tr)),
                           batch_size=bs, shuffle=True, drop_last=True)
    vl_loader = DataLoader(TensorDataset(torch.FloatTensor(X_v), torch.LongTensor(y_v)), batch_size=bs)
    best_f1, history = 0.0, {"train_loss": [], "val_f1": [], "val_acc": []}
    for epoch in range(epochs):
        model.train(); tloss = 0.0
        for xb, yb in tr_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); loss = crit(model(xb), yb); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0); opt.step()
            tloss += loss.item()
        sched.step(); model.eval()
        preds, all_y = [], []
        with torch.no_grad():
            for xb, yb in vl_loader:
                all_y.extend(yb.numpy()); preds.extend(model(xb.to(device)).argmax(1).cpu().numpy())
        vf1 = f1_score(all_y, preds, average="macro")
        vacc = accuracy_score(all_y, preds)
        history["train_loss"].append(tloss / len(tr_loader))
        history["val_f1"].append(vf1); history["val_acc"].append(vacc)
        if vf1 > best_f1:
            best_f1 = vf1
            torch.save(model.state_dict(), f"models/best_{name}.pt")
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1:3d}/{epochs} | Loss: {history['train_loss'][-1]:.4f} | Val F1: {vf1:.4f}")
    print(f"  Best Val F1: {best_f1:.4f}")
    return model, history

mlp = MLPClassifier(input_dim=480)
print("Training MLP...")
mlp, mlp_history = train_model(mlp, X_train, y_train, X_val, y_val, name="mlp")

In [ ]:
# CELL 8: MLP test set evaluation
mlp.load_state_dict(torch.load("models/best_mlp.pt", map_location=device))
mlp.eval()
with torch.no_grad():
    mlp_preds = mlp(torch.FloatTensor(X_test).to(device)).argmax(1).cpu().numpy()
mlp_result = {
    "accuracy": accuracy_score(y_test, mlp_preds),
    "f1_macro": f1_score(y_test, mlp_preds, average="macro"),
    "report": classification_report(y_test, mlp_preds, output_dict=True),
    "predictions": mlp_preds,
}
print(f"=== MLP TEST ===  Accuracy: {mlp_result['accuracy']:.4f}  F1: {mlp_result['f1_macro']:.4f}")
print(classification_report(y_test, mlp_preds, target_names=EC_CLASSES))

## Step 7: Hierarchical EC Classifier (DEEPre-inspired)

Level-by-level prediction:
- Level 1: Main class (EC 1-6)
- Level 2: Subclass per class (e.g. EC 1.1, 1.2, ..., 1.28)

Loss = Level1 CE + Level2 CE (only active class head contributes per sample)

In [ ]:
# CELL 9: Hierarchical EC classifier
EC_SUBCLASS_COUNTS = {1: 28, 2: 26, 3: 13, 4: 8, 5: 6, 6: 7}

class HierarchicalECClassifier(nn.Module):
    def __init__(self, input_dim=480, hidden_dim=256, dropout=0.3):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.BatchNorm1d(hidden_dim // 2), nn.ReLU(inplace=True), nn.Dropout(dropout),
        )
        self.level1_head = nn.Linear(hidden_dim // 2, 6)
        self.level2_heads = nn.ModuleList(
            [nn.Linear(hidden_dim // 2, count) for count in EC_SUBCLASS_COUNTS.values()]
        )

    def forward(self, x):
        feat = self.backbone(x)
        return self.level1_head(feat), [h(feat) for h in self.level2_heads]

    def predict(self, x):
        feat = self.backbone(x)
        mc = self.level1_head(feat).argmax(1)
        sc = []
        for i in range(x.shape[0]):
            sc.append(self.level2_heads[mc[i].item()](feat[i:i+1]).argmax(1).item() + 1)
        sc = torch.tensor(sc, device=x.device)
        ec_str = [f"{mc[i].item()+1}.{sc[i]}.-.-" for i in range(x.shape[0])]
        return mc, sc, ec_str


hier = HierarchicalECClassifier(input_dim=480).to(device)
opt_h = torch.optim.AdamW(hier.parameters(), lr=1e-3, weight_decay=1e-4)
sched_h = torch.optim.lr_scheduler.CosineAnnealingLR(opt_h, T_max=50)
l1_crit = nn.CrossEntropyLoss()
l2_criteria = [nn.CrossEntropyLoss() for _ in range(6)]

tr_loader = DataLoader(TensorDataset(torch.FloatTensor(X_train), torch.LongTensor(y1_train), torch.LongTensor(y2_train)),
                       batch_size=64, shuffle=True, drop_last=True)
vl_loader = DataLoader(TensorDataset(torch.FloatTensor(X_val), torch.LongTensor(y1_val), torch.LongTensor(y2_val)), batch_size=64)

print("Training Hierarchical EC Classifier...")
best_h_f1 = 0.0
hier_hist = {"train_loss": [], "val_f1": []}
for epoch in range(50):
    hier.train(); tloss = 0.0
    for xb, y1b, y2b in tr_loader:
        xb, y1b, y2b = xb.to(device), y1b.to(device), y2b.to(device)
        opt_h.zero_grad()
        l1_logits, l2_logits_list = hier(xb)
        loss = l1_crit(l1_logits, y1b)
        for cls in range(6):
            mask = (y1b == cls)
            if mask.sum() == 0: continue
            loss = loss + l2_criteria[cls](l2_logits_list[cls][mask], y2b[mask])
        loss.backward()
        nn.utils.clip_grad_norm_(hier.parameters(), max_norm=1.0)
        opt_h.step(); tloss += loss.item()
    sched_h.step(); hier.eval()
    vp, vy = [], []
    with torch.no_grad():
        for xb, y1b, _ in vl_loader:
            vy.extend(y1b.numpy()); vp.extend(hier(xb.to(device))[0].argmax(1).cpu().numpy())
    vf1 = f1_score(vy, vp, average="macro")
    hier_hist["train_loss"].append(tloss / len(tr_loader))
    hier_hist["val_f1"].append(vf1)
    if vf1 > best_h_f1: best_h_f1 = vf1; torch.save(hier.state_dict(), "models/best_hierarchical.pt")
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"  Epoch {epoch+1:3d}/50 | Loss: {hier_hist['train_loss'][-1]:.4f} | Val F1: {vf1:.4f}")
print(f"  Best Val F1: {best_h_f1:.4f}")

In [ ]:
# CELL 10: Hierarchical EC test set evaluation
hier.load_state_dict(torch.load("models/best_hierarchical.pt", map_location=device))
hier.eval()
with torch.no_grad():
    hier_mc, hier_sc, hier_ec = hier.predict(torch.FloatTensor(X_test).to(device))
    hier_mc = hier_mc.cpu().numpy()
hier_result = {
    "accuracy": accuracy_score(y1_test, hier_mc),
    "f1_macro": f1_score(y1_test, hier_mc, average="macro"),
    "report": classification_report(y1_test, hier_mc, output_dict=True),
    "predictions": hier_mc,
}
print(f"=== HIERARCHICAL EC TEST ===  Accuracy: {hier_result['accuracy']:.4f}  F1: {hier_result['f1_macro']:.4f}")
print(classification_report(y1_test, hier_mc, target_names=EC_CLASSES))
print("Sample EC predictions:")
for i in range(min(8, len(hier_ec))):
    print(f"  True: EC{y1_test[i]+1}.-.-  Pred: {hier_ec[i]}")

## Step 8: Multi-task EC + GO Model

Shared backbone with two heads:
- Primary: EC main-class (6 classes, CrossEntropy)
- Auxiliary: GO term prediction (20 terms, BCEWithLogits)

The GO head provides regularization and forces the shared representation to capture broader functional signal.

In [ ]:
# CELL 11: Multi-task EC+GO model

class MultiTaskECGO(nn.Module):
    def __init__(self, input_dim=480, hidden_dim=256, num_go=20, dropout=0.3):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.BatchNorm1d(hidden_dim // 2), nn.ReLU(inplace=True), nn.Dropout(dropout),
        )
        self.ec_head = nn.Linear(hidden_dim // 2, 6)
        self.go_head = nn.Linear(hidden_dim // 2, num_go)
    def forward(self, x):
        f = self.backbone(x)
        return self.ec_head(f), self.go_head(f)

has_go = y_go_train.sum() > 0
mt_hist = {"train_loss": [], "val_f1": []}
mt_result = None

if has_go:
    mt = MultiTaskECGO(input_dim=480, num_go=len(TOP_GO)).to(device)
    opt_m = torch.optim.AdamW(mt.parameters(), lr=1e-3, weight_decay=1e-4)
    sched_m = torch.optim.lr_scheduler.CosineAnnealingLR(opt_m, T_max=50)
    ec_crit = nn.CrossEntropyLoss(); go_crit = nn.BCEWithLogitsLoss()
    GO_WEIGHT = 0.3

    tr_loader = DataLoader(TensorDataset(torch.FloatTensor(X_train), torch.LongTensor(y_train), torch.FloatTensor(y_go_train)),
                           batch_size=64, shuffle=True, drop_last=True)
    vl_loader = DataLoader(TensorDataset(torch.FloatTensor(X_val), torch.LongTensor(y_val), torch.FloatTensor(y_go_val)), batch_size=64)

    best_mt_f1 = 0.0
    print(f"Training Multi-task EC+GO (GO weight={GO_WEIGHT})...")
    for epoch in range(50):
        mt.train(); tloss = 0.0
        for xb, yb_ec, yb_go in tr_loader:
            xb, yb_ec, yb_go = xb.to(device), yb_ec.to(device), yb_go.to(device)
            opt_m.zero_grad()
            ec_logits, go_logits = mt(xb)
            loss = ec_crit(ec_logits, yb_ec) + GO_WEIGHT * go_crit(go_logits, yb_go)
            loss.backward()
            nn.utils.clip_grad_norm_(mt.parameters(), max_norm=1.0); opt_m.step()
            tloss += loss.item()
        sched_m.step(); mt.eval()
        vp, vy = [], []
        with torch.no_grad():
            for xb, yb_ec, _ in vl_loader:
                vy.extend(yb_ec.numpy()); vp.extend(mt(xb.to(device))[0].argmax(1).cpu().numpy())
        vf1 = f1_score(vy, vp, average="macro")
        mt_hist["train_loss"].append(tloss / len(tr_loader))
        mt_hist["val_f1"].append(vf1)
        if vf1 > best_mt_f1: best_mt_f1 = vf1; torch.save(mt.state_dict(), "models/best_multitask.pt")
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1:3d}/50 | Loss: {mt_hist['train_loss'][-1]:.4f} | Val F1: {vf1:.4f}")
    print(f"  Best Val F1: {best_mt_f1:.4f}")

    # Evaluate
    mt.load_state_dict(torch.load("models/best_multitask.pt", map_location=device)); mt.eval()
    with torch.no_grad():
        ec_l, go_l = mt(torch.FloatTensor(X_test).to(device))
        mt_preds = ec_l.argmax(1).cpu().numpy()
        mt_go_preds = (torch.sigmoid(go_l) > 0.5).cpu().numpy()
    mt_result = {
        "accuracy": accuracy_score(y_test, mt_preds),
        "f1_macro": f1_score(y_test, mt_preds, average="macro"),
        "go_f1_micro": f1_score(y_go_test, mt_go_preds, average="micro"),
        "report": classification_report(y_test, mt_preds, output_dict=True),
        "predictions": mt_preds,
    }
    print(f"\n=== MULTI-TASK EC+GO TEST ===")
    print(f"EC Accuracy: {mt_result['accuracy']:.4f}  EC F1: {mt_result['f1_macro']:.4f}  GO F1 micro: {mt_result['go_f1_micro']:.4f}")
    print(classification_report(y_test, mt_preds, target_names=EC_CLASSES))
else:
    print("Skipping Multi-task training: GO annotations not available in downloaded data.")

## Step 9: Results & Figures

In [ ]:
# CELL 12: Model comparison bar chart (all models)
all_results = {}
for name, r in sklearn_results.items():
    all_results[name] = {"acc": r["accuracy"], "f1": r["f1_macro"]}
all_results["ESM-2 + MLP"] = {"acc": mlp_result["accuracy"], "f1": mlp_result["f1_macro"]}
all_results["Hierarchical EC"] = {"acc": hier_result["accuracy"], "f1": hier_result["f1_macro"]}
if mt_result:
    all_results["Multi-task EC+GO"] = {"acc": mt_result["accuracy"], "f1": mt_result["f1_macro"]}

names = sorted(all_results.keys(), key=lambda n: all_results[n]["f1"], reverse=True)
f1s = [all_results[n]["f1"] for n in names]
accs = [all_results[n]["acc"] for n in names]

fig, ax = plt.subplots(figsize=(11, 6))
colors = plt.cm.viridis(np.linspace(0.25, 0.9, len(names)))
x = np.arange(len(names))
ax.bar(x, f1s, color=colors, width=0.5, label="Macro F1")
ax.bar(x, accs, color=colors, width=0.25, alpha=0.5, label="Accuracy")
for i, (f1, acc) in enumerate(zip(f1s, accs)):
    ax.text(i, f1 + 0.01, f"{f1:.3f}", ha="center", fontsize=9, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(names, rotation=20, ha="right")
ax.set_title("Model Comparison — Enzyme EC Classification")
ax.set_ylabel("Score"); ax.set_ylim(0, 1.05); ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("results/figures/model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# CELL 13: Confusion matrix (MLP)
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, mlp_result["predictions"])
cm_pct = cm.astype("float") / cm.sum(axis=1)[:, None] * 100
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=EC_CLASSES, yticklabels=EC_CLASSES, ax=axes[0])
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True"); axes[0].set_title("Confusion Matrix (Counts)")
sns.heatmap(cm_pct, annot=True, fmt=".1f", cmap="YlOrRd", xticklabels=EC_CLASSES, yticklabels=EC_CLASSES, ax=axes[1])
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("True"); axes[1].set_title("Confusion Matrix (%)")
plt.suptitle("ESM-2 + MLP", fontsize=15, y=1.02); plt.tight_layout()
plt.savefig("results/figures/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# CELL 14: Training curves for MLP
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
epochs_range = range(1, len(mlp_history["train_loss"]) + 1)
axes[0].plot(epochs_range, mlp_history["train_loss"], color="#2196F3", linewidth=2, label="Train Loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].set_title("Training Loss"); axes[0].grid(True, alpha=0.3)
axes[1].plot(epochs_range, mlp_history["val_f1"], color="#4CAF50", linewidth=2, label="Val Macro F1")
axes[1].plot(epochs_range, mlp_history["val_acc"], color="#FF9800", linewidth=2, linestyle="--", label="Val Accuracy")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Score"); axes[1].set_title("Validation Performance")
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("results/figures/training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# CELL 15: Save all results to JSON
results = {
    "sklearn_baselines": {name: {"accuracy": float(r["accuracy"]), "f1_macro": float(r["f1_macro"])}
                          for name, r in sklearn_results.items()},
    "pytorch_models": {
        "ESM-2 + MLP": {"accuracy": float(mlp_result["accuracy"]), "f1_macro": float(mlp_result["f1_macro"])},
        "Hierarchical EC": {"accuracy": float(hier_result["accuracy"]), "f1_macro": float(hier_result["f1_macro"])},
    },
    "training_history": {
        "mlp": mlp_history, "hierarchical": hier_hist, "multitask": mt_hist,
    },
}
if mt_result:
    results["pytorch_models"]["Multi-task EC+GO"] = {
        "accuracy": float(mt_result["accuracy"]), "f1_macro": float(mt_result["f1_macro"]),
        "go_f1_micro": float(mt_result.get("go_f1_micro", 0)),
    }

with open("results/experiment_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("\n=== FINAL SUMMARY ===")
print(f"{'Model':25s}  {'Accuracy':>10s}  {'Macro F1':>10s}")
print("-" * 50)
for name in sorted(all_results.keys(), key=lambda n: all_results[n]["f1"], reverse=True):
    r = all_results[name]
    print(f"{name:25s}  {r['acc']:10.4f}  {r['f1']:10.4f}")
print("\nSaved: results/experiment_results.json")

In [ ]:
# CELL 16: Zip all results for download
!zip -r results.zip results models enzymes_swissprot.csv esm2_embeddings.npy labels.npy
print("\nDownload results.zip from the Files panel (left sidebar).")